In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# from kret_studies import *
# from kret_studies.notebook import *
# from kret_studies.complex import *

# logger = get_notebook_logger()

In [3]:
from waymo_agent.notebook_imports import *

In [4]:
from waymo_agent import *
from waymo_agent.osmnx import *
from waymo_agent.data_classes import *
from waymo_agent.action_heuristic import *
from waymo_agent.graph_env import *
from waymo_agent.simulation import *
from waymo_agent.action_heuristic.heuristic_simple import PricingAgent, DispatchAgent, RepositionAgent
from waymo_agent.models.ppo_model import RideShareActorCritic
from waymo_agent.graph_env.ENV import RideShareEnv
from waymo_agent.models.ppo_model import *

In [5]:
from waymo_agent.models.ppo_model import *
from waymo_agent.models.evaluate import *
from waymo_agent.models.train_utils import *
from waymo_agent.models.torch_np_utils import *
from waymo_agent.models.torch_np_utils import _flat_obs

In [6]:
from kret_sandbox.VIS import dtt
from kret_sandbox.exp_decay import exp_decay_half_life, get_gamma_from_half_life

In [7]:
def get_obs_tuple(env: RideShareEnv):
    veh = env.observation_curr["vehicles"]
    req = env.observation_curr["pending_requests"]
    rides = env.observation_curr["active_rides"]
    return veh, req, rides


def get_obs_tuple_from_obs(obs: ObservationDict):
    veh = obs["vehicles"]
    req = obs["pending_requests"]
    rides = obs["active_rides"]
    return veh, req, rides

## Env Init

In [8]:
env_cfg = EnvConfig(
    max_episode_steps=60 * 1,
    vehicle_per_node=0.10,
    lambda_per_node=0.05,
    max_pending_requests=100,
    max_new_requests_per_step=20,
)
plt_cfg = PlotConfig()
env_model = RideShareEnv(env_cfg, plt_cfg)
env = env_model

Assigned lambda values to nodes. Total lambda: 39.1000 (target: 39.1000)


In [9]:
env_model.config.no_action_id

-1

In [10]:
obs, info = env_model.reset()

In [11]:
veh, req, rides = get_obs_tuple_from_obs(obs)

In [12]:
veh_enr, req_enr, rides_enr = get_obs_tuple(env_model)

In [13]:
veh_enr.head(4)

,vehicle_id,loc_x_norm,loc_y_norm,battery,status,ride_id
0,0,0.130417,-0.167523,0.952531,0,-1
1,1,-0.035172,0.198614,0.965102,0,-1
2,2,0.075608,-0.565520,0.995456,0,-1
3,3,-0.427512,-0.398919,0.721437,0,-1


# Test Subheads


In [14]:
GAMMA = get_gamma_from_half_life(env_cfg.max_episode_steps // 2)
round(GAMMA, 5)

0.97716

## Pricing Head

In [15]:
obs_np, _ = env_model.reset(seed=0)
obs_t = obs_pd_to_torch(obs_np)  # dict[str, Tensor] on DEVICE (if your converter does that)

# --- build encoder + head (standalone) ---
# Use same obs_dim calculation RideShareActorCritic uses:
x = _flat_obs(obs_t)  # (obs_dim,)
obs_dim = int(x.numel())

hidden = 256
enc = SharedEncoder(obs_dim, hidden).to(DEVICE).eval()
head = PricingHead(hidden, env_model.config).to(DEVICE).eval()

with torch.no_grad():
    h = enc(x)  # (hidden,)
    # PricingHead expects h shaped like (hidden,) or (B, hidden). We'll enforce (B, hidden).
    if h.ndim == 1:
        hB = h.unsqueeze(0)  # (1, hidden)
    else:
        hB = h

    # 1) sample action
    a = head.act(hB, obs=obs_t, deterministic=False)  # (1, max_pending)
    assert a.shape[-1] == env_model.config.max_pending_requests
    assert torch.isfinite(a).all()
    assert (a > 0).all()

    # 2) logprob/entropy should be finite
    logp, ent = head.log_prob_and_entropy(hB, a, obs=obs_t)
    assert logp.shape == (1,)
    assert ent.shape == (1,)
    assert torch.isfinite(logp).all()
    assert torch.isfinite(ent).all()

    # 3) mask behavior: where pricing_mask==0, action should be eps_price
    if "pricing_mask" in obs_t:
        pm = obs_t["pricing_mask"].to(device=a.device, dtype=a.dtype)  # (50,) likely
        # broadcast pm to (1, 50)
        pmB = pm.unsqueeze(0) if pm.ndim == 1 else pm
        eps = head.eps_price
        masked = pmB <= 0.0
        if masked.any():
            assert torch.allclose(
                a[masked], torch.full_like(a[masked], eps)
            ), f"Masked prices not set to eps_price={eps}"

print("PricingHead smoke test: PASS ✅")

PricingHead smoke test: PASS ✅


## Reposition Head

In [16]:
obs_np, _ = env.reset(seed=0)
obs_t = obs_pd_to_torch(obs_np)  # dict[str, Tensor] on DEVICE

# --- Build encoder + head (standalone) ---
x = _flat_obs(obs_t)  # (obs_dim,)
obs_dim = int(x.numel())

hidden = 256
enc = SharedEncoder(obs_dim, hidden).to(DEVICE).eval()
head = RepositionHead(hidden, env.num_vehicles).to(DEVICE).eval()

with torch.no_grad():
    h = enc(x)  # (hidden,)
    hB = h.unsqueeze(0)  # (1, hidden)

    # 1) sample action
    a = head.act(hB, obs=obs_t, deterministic=False)  # (1, num_veh, 2)
    assert a.shape == (1, env.num_vehicles, 2), a.shape
    assert torch.isfinite(a).all()
    assert (a >= -1.0).all() and (a <= 1.0).all()

    # 2) logprob/entropy finite + shapes
    logp, ent = head.log_prob_and_entropy(hB, a, obs=obs_t)
    assert logp.shape == (1,), logp.shape
    assert ent.shape == (1,), ent.shape
    assert torch.isfinite(logp).all()
    assert torch.isfinite(ent).all()

    # 3) mask behavior:
    # make a fake mask where only first half vehicles are "idle"
    m = obs_t["dispatch_mask"].clone()
    m[:] = 0.0
    m[: env.num_vehicles // 2] = 1.0

    obs_t_masked = dict(obs_t)
    obs_t_masked["dispatch_mask"] = m

    a2 = head.act(hB, obs=obs_t_masked, deterministic=False)  # (1, num_veh, 2)
    # vehicles where mask==0 should be exactly 0 (per your implementation)
    mB = m.unsqueeze(0).to(device=a2.device, dtype=a2.dtype)  # (1, num_veh)
    # On masked vehicles, a2 should be ~0 for both coords
    assert torch.allclose(a2 * (1.0 - mB.unsqueeze(-1)), torch.zeros_like(a2)), "Masked reposition actions not zeroed"

    # logprob should still be finite under mask
    logp2, ent2 = head.log_prob_and_entropy(hB, a2, obs=obs_t_masked)
    assert torch.isfinite(logp2).all()
    assert torch.isfinite(ent2).all()

print("RepositionHead smoke test: PASS ✅")

RepositionHead smoke test: PASS ✅


## Dispatch Head

In [17]:
obs_np, _ = env.reset(seed=0)
obs_t = obs_pd_to_torch(obs_np)

num_veh = env.num_vehicles
max_pending = env.config.max_pending_requests

# ------------------------------------------------------------
# Build encoder + DispatchHead (standalone)
# ------------------------------------------------------------
x = _flat_obs(obs_t)  # (obs_dim,)
obs_dim = int(x.numel())

hidden = 256
enc = SharedEncoder(obs_dim, hidden).to(DEVICE).eval()
head = DispatchHead(hidden, max_pending, num_veh).to(DEVICE).eval()

with torch.no_grad():
    # ensure batch dimension
    h = enc(x.unsqueeze(0))  # (1, hidden)

    # --------------------------------------------------------
    # 1) Sample stochastic dispatch action
    # --------------------------------------------------------
    a = head.act(h, obs=obs_t, deterministic=False)
    assert a.shape == (1, max_pending), a.shape
    assert a.dtype in (torch.int64, torch.int32)
    assert torch.isfinite(a).all()

    # dispatch values must be in [-1 .. num_veh-1]
    assert (a >= -1).all()
    assert (a < num_veh).all()

    # --------------------------------------------------------
    # 2) log_prob / entropy finite
    # --------------------------------------------------------
    logp, ent = head.log_prob_and_entropy(h, a, obs=obs_t)
    assert logp.shape == (1,)
    assert ent.shape == (1,)
    assert torch.isfinite(logp).all()
    assert torch.isfinite(ent).all()

    # --------------------------------------------------------
    # 3) Mask semantics: forbid all vehicles
    # --------------------------------------------------------
    obs_t_masked = dict(obs_t)
    obs_t_masked["dispatch_mask"] = torch.zeros_like(obs_t["dispatch_mask"])

    a_masked = head.act(h, obs=obs_t_masked, deterministic=False)
    # All dispatches must be NO-ACTION (-1)
    assert (a_masked == -1).all(), "DispatchHead failed mask semantics: non-idle vehicle assigned"

    logp_m, ent_m = head.log_prob_and_entropy(h, a_masked, obs=obs_t_masked)
    assert torch.isfinite(logp_m).all()
    assert torch.isfinite(ent_m).all()

    # --------------------------------------------------------
    # 4) Partial mask: only allow first half vehicles
    # --------------------------------------------------------
    m = torch.zeros_like(obs_t["dispatch_mask"])
    m[: num_veh // 2] = 1.0
    obs_t_half = dict(obs_t)
    obs_t_half["dispatch_mask"] = m

    a_half = head.act(h, obs=obs_t_half, deterministic=False)
    a_np = a_half.squeeze(0).cpu().numpy()

    # Any actual vehicle assignment must be within allowed range
    for d in a_np:
        if d >= 0:
            assert d < num_veh // 2, f"Assigned masked vehicle {d} with dispatch_mask=0"

    # --------------------------------------------------------
    # 5) Deterministic policy sanity
    # --------------------------------------------------------
    a_det = head.act(h, obs=obs_t, deterministic=True)
    assert a_det.shape == (1, max_pending)
    assert torch.isfinite(a_det).all()

print("DispatchHead smoke test: PASS ✅")

DispatchHead smoke test: PASS ✅


In [18]:
import numpy as np

from waymo_agent.graph_env.ENV import RideShareEnv, EnvConfig
from waymo_agent.data_classes.requests import RequestStatusEnum
from waymo_agent.data_classes.vehicles import VehicleStatusEnum


def make_action(env, *, prices=None, dispatch=None, reposition=None):
    if prices is None:
        prices = np.zeros(env.config.max_pending_requests, dtype=np.float64)
    if dispatch is None:
        dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    if reposition is None:
        reposition = np.zeros((env.num_vehicles, 2), dtype=np.float32)
    return {"prices": prices, "dispatch": dispatch, "reposition": reposition}


def find_req_row_by_id(req_df, req_id: int) -> int | None:
    rows = np.flatnonzero((req_df["request_id"].to_numpy() == req_id))
    return int(rows[0]) if rows.size else None


obs, _ = env.reset(seed=0)

accepted_req_id = None

# 1) Drive until we see an ACCEPTED request
for _ in range(200):
    obs, r, term, trunc, info = env.step(make_action(env))
    req = env.observation_curr["pending_requests"]
    f_acc = req["status"].to_numpy() == RequestStatusEnum.ACCEPTED.value
    if f_acc.any():
        # record the *stable* request_id, not row index
        accepted_req_id = int(req.loc[f_acc, "request_id"].iloc[0])
        break
    if term or trunc:
        break

assert accepted_req_id is not None, "Never observed an ACCEPTED request."

# 2) On the current step, find its CURRENT row index (needed for the action)
req = env.observation_curr["pending_requests"]
req_row = find_req_row_by_id(req, accepted_req_id)
assert req_row is not None, "ACCEPTED request disappeared before dispatch."

# 3) Choose an idle vehicle by row index (dispatch_mask is in row-index space)
veh = env.observation_curr["vehicles"]
dispatch_mask = np.asarray(env.observation_curr["dispatch_mask"], dtype=bool)
idle_idxs = np.flatnonzero(dispatch_mask)
assert idle_idxs.size > 0, "No IDLE vehicles available."

veh_row = int(idle_idxs[0])

# sanity
assert int(req.iloc[req_row]["status"]) == RequestStatusEnum.ACCEPTED.value
assert int(veh.iloc[veh_row]["status"]) == VehicleStatusEnum.IDLE.value

pre_veh_rideid = int(veh.iloc[veh_row]["ride_id"])

# 4) Dispatch: set exactly one row of the dispatch vector
dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
dispatch[req_row] = veh_row

obs2, r2, term2, trunc2, info2 = env.step(make_action(env, dispatch=dispatch))

# 5) Validate by looking up the request again by request_id (row may have moved!)
req2 = env.observation_curr["pending_requests"]
req2_row = find_req_row_by_id(req2, accepted_req_id)
assert req2_row is not None, "Request vanished after dispatch (trim/reject?)"

post_status = int(req2.iloc[req2_row]["status"])
assert (
    post_status == RequestStatusEnum.ASSIGNED.value
), f"Expected ACCEPTED -> ASSIGNED for request_id={accepted_req_id}, got status={post_status}."

veh2 = env.observation_curr["vehicles"]
post_veh_status = int(veh2.iloc[veh_row]["status"])
post_veh_rideid = int(veh2.iloc[veh_row]["ride_id"])

assert post_veh_status != VehicleStatusEnum.IDLE.value, "Vehicle did not leave IDLE after dispatch."
assert post_veh_rideid != pre_veh_rideid, "Vehicle ride_id did not change after dispatch."

print("Dispatch end-to-end test (request_id-stable): PASS ✅")
print(f"  request_id={accepted_req_id} row {req_row} -> {req2_row}, status={post_status}")
print(f"  vehicle row={veh_row}, ride_id {pre_veh_rideid} -> {post_veh_rideid}, status={post_veh_status}")

Dispatch end-to-end test (request_id-stable): PASS ✅
  request_id=119 row 20 -> 37, status=3
  vehicle row=0, ride_id -1 -> 119, status=2


In [19]:
import numpy as np

from waymo_agent.data_classes.requests import RequestStatusEnum
from waymo_agent.data_classes.vehicles import VehicleStatusEnum


def make_action(env, *, prices=None, dispatch=None, reposition=None):
    if prices is None:
        prices = np.zeros(env.config.max_pending_requests, dtype=np.float64)
    if dispatch is None:
        dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    if reposition is None:
        reposition = np.zeros((env.num_vehicles, 2), dtype=np.float32)
    return {"prices": prices, "dispatch": dispatch, "reposition": reposition}


def find_rows_by_ids(req_df, ids):
    rid = req_df["request_id"].to_numpy()
    out = {}
    for i in ids:
        hits = np.flatnonzero(rid == i)
        out[i] = int(hits[0]) if hits.size else None
    return out


# --- 1) Drive until >=2 ACCEPTED requests exist ---
accepted_ids = None
for _ in range(500):
    _obs, _r, term, trunc, _info = env.step(make_action(env))
    req = env.observation_curr["pending_requests"]
    f_acc = req["status"].to_numpy() == RequestStatusEnum.ACCEPTED.value
    if f_acc.sum() >= 2:
        accepted_ids = req.loc[f_acc, "request_id"].iloc[:2].astype(int).to_list()
        break
    if term or trunc:
        break

assert accepted_ids is not None, "Never got 2 ACCEPTED requests."

# --- 2) Choose an idle vehicle row ---
veh = env.observation_curr["vehicles"]
dispatch_mask = np.asarray(env.observation_curr["dispatch_mask"], dtype=bool)
idle_rows = np.flatnonzero(dispatch_mask)
assert idle_rows.size > 0, "No idle vehicles."
veh_row = int(idle_rows[0])

assert int(veh.iloc[veh_row]["status"]) == VehicleStatusEnum.IDLE.value

# --- 3) Build dispatch action that assigns both requests to same vehicle ---
req = env.observation_curr["pending_requests"]
rows = find_rows_by_ids(req, accepted_ids)
assert all(v is not None for v in rows.values()), f"Accepted req vanished: {rows}"

dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
dispatch[rows[accepted_ids[0]]] = veh_row
dispatch[rows[accepted_ids[1]]] = veh_row

pre_rideid = int(env.observation_curr["vehicles"].iloc[veh_row]["ride_id"])

_obs2, _r2, _term2, _trunc2, _info2 = env.step(make_action(env, dispatch=dispatch))

# --- 4) Verify: exactly one assignment happened ---
req2 = env.observation_curr["pending_requests"]
rows2 = find_rows_by_ids(req2, accepted_ids)
assert all(v is not None for v in rows2.values()), f"Post-step req vanished: {rows2}"

statuses = [int(req2.iloc[rows2[i]]["status"]) for i in accepted_ids]
n_assigned = sum(s == RequestStatusEnum.ASSIGNED.value for s in statuses)

assert n_assigned == 1, f"Expected exactly 1 ASSIGNED, got statuses={statuses} for ids={accepted_ids}"

veh2 = env.observation_curr["vehicles"]
post_status = int(veh2.iloc[veh_row]["status"])
post_rideid = int(veh2.iloc[veh_row]["ride_id"])

assert post_status != VehicleStatusEnum.IDLE.value, "Vehicle did not leave IDLE"
assert post_rideid != pre_rideid, "Vehicle ride_id did not update"

print("Collision dispatch test: PASS ✅")
print(f"  req_ids={accepted_ids} statuses={statuses} (exactly one ASSIGNED)")
print(f"  veh_row={veh_row} status={post_status} ride_id={pre_rideid}->{post_rideid}")

Collision dispatch test: PASS ✅
  req_ids=[159, 158] statuses=[3, 1] (exactly one ASSIGNED)
  veh_row=1 status=2 ride_id=-1->159


In [20]:
import numpy as np
import torch

from waymo_agent.graph_env.ENV import RideShareEnv, EnvConfig
from waymo_agent.models.torch_np_utils import obs_pd_to_torch, _flat_obs
from waymo_agent.constants import DEVICE_TORCH_STR
from waymo_agent.models.ppo_model import RideShareActorCritic  # adjust if you moved it

DEVICE = torch.device(DEVICE_TORCH_STR)


def action_torch_to_numpy(act_t: dict[str, torch.Tensor]) -> dict[str, np.ndarray]:
    # Keep dtypes consistent with your gym spaces
    out = {}
    out["prices"] = act_t["prices"].detach().cpu().numpy().astype(np.float64)
    out["dispatch"] = act_t["dispatch"].detach().cpu().numpy().astype(np.int64)
    out["reposition"] = act_t["reposition"].detach().cpu().numpy().astype(np.float32)
    return out


def assert_finite_tensor(name: str, x: torch.Tensor):
    assert torch.isfinite(x).all(), f"{name} has non-finite values: {x}"


# ----------------------------
# Setup env/model
# ----------------------------
# env = RideShareEnv(EnvConfig())
model = RideShareActorCritic(env).to(DEVICE).eval()

obs_np, _ = env.reset(seed=0)

T = 25  # short rollout

for t in range(T):
    obs_t = obs_pd_to_torch(obs_np)

    # 0) obs finite check
    x = _flat_obs(obs_t)
    assert_finite_tensor(f"flat_obs[t={t}]", x)

    # 1) sample action
    with torch.no_grad():
        act_t = model.act(obs_t, deterministic=False)

    # 2) action structure checks
    for k in ("prices", "dispatch", "reposition"):
        assert k in act_t, f"Missing action key: {k}"
        assert isinstance(act_t[k], torch.Tensor), f"{k} is not a tensor"

    prices = act_t["prices"]
    dispatch = act_t["dispatch"]
    reposition = act_t["reposition"]

    assert prices.ndim == 1 and prices.shape[0] == env.config.max_pending_requests, prices.shape
    assert dispatch.ndim == 1 and dispatch.shape[0] == env.config.max_pending_requests, dispatch.shape
    assert reposition.ndim == 2 and reposition.shape == (env.num_vehicles, 2), reposition.shape

    assert_finite_tensor(f"prices[t={t}]", prices)
    assert_finite_tensor(f"dispatch[t={t}]", dispatch.to(torch.float32))
    assert_finite_tensor(f"reposition[t={t}]", reposition)

    # 3) validity constraints
    # prices must be positive (PricingHead clamps to eps_price=1.0)
    assert (prices >= 1e-6).all(), f"prices has non-positive entries at t={t}"

    # reposition in [-1, 1]
    assert (reposition >= -1.0).all() and (reposition <= 1.0).all(), f"reposition out of bounds at t={t}"

    # dispatch in {-1, 0..num_vehicles-1}
    assert (dispatch >= -1).all(), f"dispatch < -1 at t={t}"
    assert (dispatch < env.num_vehicles).all(), f"dispatch >= num_vehicles at t={t}"

    # 4) mask constraints: if dispatch assigns a vehicle, it must be idle per dispatch_mask
    dm = obs_t["dispatch_mask"].detach().cpu().numpy().astype(bool)  # shape (num_vehicles,)
    disp_np = dispatch.detach().cpu().numpy()
    assigned = disp_np[disp_np >= 0].astype(int)
    if assigned.size > 0:
        # every chosen vehicle index must be allowed by the mask
        assert dm[assigned].all(), f"dispatch assigned masked/busy vehicle at t={t}: {assigned[~dm[assigned]]}"

    # pricing mask: masked entries should still be valid prices (your head sets eps_price there)
    pm = obs_t["pricing_mask"]
    assert_finite_tensor(f"pricing_mask[t={t}]", pm)

    # 5) log_prob/entropy/value must be finite
    with torch.no_grad():
        out = model.log_prob_and_entropy(obs_t, act_t)

    # support either (logp, ent, v) or dict-like returns
    if isinstance(out, tuple) and len(out) == 3:
        logp_t, ent_t, v_t = out
    else:
        # try common dict keys
        logp_t = out["logp"]
        ent_t = out["ent"]
        v_t = out["v"]

    assert_finite_tensor(f"logp[t={t}]", logp_t)
    assert_finite_tensor(f"ent[t={t}]", ent_t)
    assert_finite_tensor(f"value[t={t}]", v_t)

    # 6) step env + reward finite
    act_np = action_torch_to_numpy(act_t)
    obs_np, r, term, trunc, info = env.step(act_np)
    assert np.isfinite(r), f"reward non-finite at t={t}: {r}"

    if term or trunc:
        break

print(f"PPO rollout numerics test: PASS ✅  (ran {t+1} steps)")

PPO rollout numerics test: PASS ✅  (ran 25 steps)


In [21]:
dtt([req, req2], 50, "head", num_cols=1)

,request_id,request_dt,cust_id,cust_bias,cust_temperature,est_cost,price,max_wait_time,wait_time,status,pickup_node_id,pickup_x_norm,pickup_y_norm,dropoff_node_id,dropoff_x_norm,dropoff_y_norm,route_nodes,curr_start_node,curr_end_node,route_dist_on_edge,distance_meters
,int64,datetime64[ns],int64,float64,float64,float64,float64,timedelta64[ns],timedelta64[ns],int64,int64,float64,float64,int64,float64,float64,object,int64,int64,float64,float64
0,179,2025-01-03 07:13:31.733894,13517,-0.148,2.301,4.286,NaN,0 days 00:15:00,0 days 00:00:00,0,11017187937,-0.295,0.145,42430237,-0.311,-0.334,"[11017187937, 42430688, 42430685, 7280430895, 9019680388, 42437371, 42455751, 42443975, 42452975, 42437670, 1701844618, 427841071, 42434465, 42437688, 42437909, 42435624, 42432214, 42430828, 42430237]",11017187937,42430688,0.0,4286.308
1,178,2025-01-03 07:13:31.733894,14551,-2.834,1.676,6.502,NaN,0 days 00:15:00,0 days 00:00:00,0,11003445421,0.142,-0.112,4145735059,-0.473,-0.831,"[11003445421, 12154055788, 42422899, 42445390, 42445417, 42448707, 42440743, 5798966629, 7480301864, 486868873, 3212472843, 42437996, 42437990, 7480301987, 42457821, 3212472978, 42444123, 1538237325, 42444108, 278609760, 12299314857, 12299314860, 42433218, 42421951, 370880758, 205020852, 4145735059]",11003445421,12154055788,0.0,6502.182
2,177,2025-01-03 07:13:31.733894,1755,-2.998,1.616,9.655,NaN,0 days 00:15:00,0 days 00:00:00,0,6185259892,0.552,0.387,42424354,-0.437,-0.535,"[6185259892, 42428037, 42428020, 42428007, 42427970, 42427968, 42427965, 371188750, 13238997597, 589099734, 42445392, 42445390, 42445417, 42448707, 42440743, 5798966629, 7480301864, 1918039877, 1918039904, 1918039864, 1918039897, 1919595915, 42428438, 42428436, 42440829, 4143859873, 4143859865, 4778174564, 42424354]",6185259892,42428037,0.0,9655.333
3,176,2025-01-03 07:13:31.733894,1486,-0.090,1.641,4.442,NaN,0 days 00:15:00,0 days 00:00:00,0,4779073680,0.556,0.437,42455929,0.151,-0.095,"[4779073680, 4779073679, 42429342, 42456060, 42436492, 42436531, 6177439749, 42448326, 2711029280, 42455963, 42436714, 42455934, 6133355290, 595105499, 42434092, 42444055, 595105472, 42455929]",4779073680,4779073679,0.0,4442.198
4,175,2025-01-03 07:13:31.733894,27669,-1.392,2.058,11.850,NaN,0 days 00:15:00,0 days 00:00:00,0,4142073861,-0.494,-0.456,42444456,0.638,0.783,"[4142073861, 42440854, 12417264049, 42424630, 42423514, 42423456, 42430143, 42428438, 1919595915, 42428447, 42428473, 4597668041, 561042190, 561042193, 561042192, 100522479, 9140654137, 100522741, 42438886, 42459098, 42436703, 42456611, 42432834, 42448317, 6177439750, 42455666, 42447230, 42436481, 42459137, 42429330, 42457660, 7076060068, 42450460, 42444456]",4142073861,42440854,0.0,11849.721
5,174,2025-01-03 07:13:31.733894,45663,-0.241,2.367,11.049,NaN,0 days 00:15:00,0 days 00:00:00,0,3785532382,0.556,0.391,370880739,-0.506,-0.835,"[3785532382, 42428052, 6185259892, 42428037, 42428020, 42428007, 42427970, 42427968, 42427965, 371188750, 13238997597, 589099734, 42445392, 42445390, 42445417, 42448707, 42440743, 5798966629, 7480301864, 486868873, 3212472843, 42437996, 42437990, 7480301987, 42451650, 1773066054, 588455742, 588455743, 42437749, 3884569931, 3884569924, 278609934, 42422283, 42426747, 370880739]",3785532382,42428052,0.0,11049.166
6,173,2025-01-03 07:13:31.733894,8757,2.666,2.351,4.235,NaN,0 days 00:15:00,0 days 00:00:00,0,587812578,0.069,0.644,42445976,0.179,0.203,"[587812578, 42428714, 42428711, 42428674, 42428653, 42428634, 9177424867, 42428595, 1061531707, 3718672443, 7802856352, 247223720, 42457788, 42447228, 42445976]",587812578,42428714,0.0,4234.599
7,172,2025-01-03 07:13:31.733894,33884,-1.742,2.221,11.478,NaN,0 days 00:15:00,0 days 00:00:00,0,373755525,0.724,0.807,42429762,-0.167,-0.562,"[373755525, 42426799, 371225088, 7641820943, 9490559521, 42450468, 4779073679, 42429342, 42456060, 42436492, 42436531, 6177439749, 42448326, 2711029280, 42455963, 42436714, 42455934, 6133355290, 5852272391, 11003445421, 12154055788, 42422899, 42445

In [22]:
import numpy as np
import pytest

from waymo_agent.graph_env.ENV import RideShareEnv, EnvConfig
from waymo_agent.data_classes.requests import RequestStatusEnum
from waymo_agent.data_classes.vehicles import VehicleStatusEnum


def make_action(env, *, prices=None, dispatch=None, reposition=None):
    if prices is None:
        prices = np.zeros(env.config.max_pending_requests, dtype=np.float64)
    if dispatch is None:
        dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    if reposition is None:
        reposition = np.zeros((env.num_vehicles, 2), dtype=np.float32)
    return {"prices": prices, "dispatch": dispatch, "reposition": reposition}


def find_req_row_by_id(req_df, req_id: int) -> int | None:
    rows = np.flatnonzero(req_df["request_id"].to_numpy() == req_id)
    return int(rows[0]) if rows.size else None


@pytest.fixture()
def env():
    e = RideShareEnv(EnvConfig())
    e.reset(seed=0)
    return e


def test_dispatch_feasible_assigns_one(env):
    accepted_req_id = None

    for _ in range(300):
        _obs, _r, term, trunc, _info = env.step(make_action(env))
        req = env.observation_curr["pending_requests"]
        f_acc = req["status"].to_numpy() == RequestStatusEnum.ACCEPTED.value
        if f_acc.any():
            accepted_req_id = int(req.loc[f_acc, "request_id"].iloc[0])
            break
        if term or trunc:
            break

    assert accepted_req_id is not None, "Never saw ACCEPTED request"

    req = env.observation_curr["pending_requests"]
    req_row = find_req_row_by_id(req, accepted_req_id)
    assert req_row is not None

    dm = np.asarray(env.observation_curr["dispatch_mask"], dtype=bool)
    idle_rows = np.flatnonzero(dm)
    assert idle_rows.size > 0
    veh_row = int(idle_rows[0])

    veh = env.observation_curr["vehicles"]
    assert int(veh.iloc[veh_row]["status"]) == VehicleStatusEnum.IDLE.value

    dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    dispatch[req_row] = veh_row

    _obs2, _r2, _term2, _trunc2, _info2 = env.step(make_action(env, dispatch=dispatch))

    req2 = env.observation_curr["pending_requests"]
    req2_row = find_req_row_by_id(req2, accepted_req_id)
    assert req2_row is not None

    post_status = int(req2.iloc[req2_row]["status"])
    assert post_status == RequestStatusEnum.ASSIGNED.value


def test_dispatch_collision_only_one_wins(env):
    accepted_ids = None
    for _ in range(600):
        _obs, _r, term, trunc, _info = env.step(make_action(env))
        req = env.observation_curr["pending_requests"]
        f_acc = req["status"].to_numpy() == RequestStatusEnum.ACCEPTED.value
        if f_acc.sum() >= 2:
            accepted_ids = req.loc[f_acc, "request_id"].iloc[:2].astype(int).to_list()
            break
        if term or trunc:
            break

    assert accepted_ids is not None

    dm = np.asarray(env.observation_curr["dispatch_mask"], dtype=bool)
    idle_rows = np.flatnonzero(dm)
    assert idle_rows.size > 0
    veh_row = int(idle_rows[0])

    req = env.observation_curr["pending_requests"]
    rows = {rid: find_req_row_by_id(req, rid) for rid in accepted_ids}
    assert all(v is not None for v in rows.values())

    dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    dispatch[rows[accepted_ids[0]]] = veh_row
    dispatch[rows[accepted_ids[1]]] = veh_row

    env.step(make_action(env, dispatch=dispatch))

    req2 = env.observation_curr["pending_requests"]
    rows2 = {rid: find_req_row_by_id(req2, rid) for rid in accepted_ids}
    assert all(v is not None for v in rows2.values())

    statuses = [int(req2.iloc[rows2[rid]]["status"]) for rid in accepted_ids]
    n_assigned = sum(s == RequestStatusEnum.ASSIGNED.value for s in statuses)
    assert n_assigned == 1, f"Expected exactly 1 assigned, got {statuses}"

In [23]:
import numpy as np
import pytest

from waymo_agent.graph_env.ENV import RideShareEnv, EnvConfig
from waymo_agent.data_classes.requests import RequestStatusEnum
from waymo_agent.data_classes.vehicles import VehicleStatusEnum


def make_action(env, *, prices=None, dispatch=None, reposition=None):
    if prices is None:
        prices = np.zeros(env.config.max_pending_requests, dtype=np.float64)
    if dispatch is None:
        dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    if reposition is None:
        reposition = np.zeros((env.num_vehicles, 2), dtype=np.float32)
    return {"prices": prices, "dispatch": dispatch, "reposition": reposition}


def find_req_row_by_id(req_df, req_id: int) -> int | None:
    rows = np.flatnonzero(req_df["request_id"].to_numpy() == req_id)
    return int(rows[0]) if rows.size else None


@pytest.fixture()
def env():
    e = RideShareEnv(EnvConfig())
    e.reset(seed=0)
    return e


def test_dispatch_feasible_assigns_one(env):
    accepted_req_id = None

    for _ in range(300):
        _obs, _r, term, trunc, _info = env.step(make_action(env))
        req = env.observation_curr["pending_requests"]
        f_acc = req["status"].to_numpy() == RequestStatusEnum.ACCEPTED.value
        if f_acc.any():
            accepted_req_id = int(req.loc[f_acc, "request_id"].iloc[0])
            break
        if term or trunc:
            break

    assert accepted_req_id is not None, "Never saw ACCEPTED request"

    req = env.observation_curr["pending_requests"]
    req_row = find_req_row_by_id(req, accepted_req_id)
    assert req_row is not None

    dm = np.asarray(env.observation_curr["dispatch_mask"], dtype=bool)
    idle_rows = np.flatnonzero(dm)
    assert idle_rows.size > 0
    veh_row = int(idle_rows[0])

    veh = env.observation_curr["vehicles"]
    assert int(veh.iloc[veh_row]["status"]) == VehicleStatusEnum.IDLE.value

    dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    dispatch[req_row] = veh_row

    _obs2, _r2, _term2, _trunc2, _info2 = env.step(make_action(env, dispatch=dispatch))

    req2 = env.observation_curr["pending_requests"]
    req2_row = find_req_row_by_id(req2, accepted_req_id)
    assert req2_row is not None

    post_status = int(req2.iloc[req2_row]["status"])
    assert post_status == RequestStatusEnum.ASSIGNED.value


def test_dispatch_collision_only_one_wins(env):
    accepted_ids = None
    for _ in range(600):
        _obs, _r, term, trunc, _info = env.step(make_action(env))
        req = env.observation_curr["pending_requests"]
        f_acc = req["status"].to_numpy() == RequestStatusEnum.ACCEPTED.value
        if f_acc.sum() >= 2:
            accepted_ids = req.loc[f_acc, "request_id"].iloc[:2].astype(int).to_list()
            break
        if term or trunc:
            break

    assert accepted_ids is not None

    dm = np.asarray(env.observation_curr["dispatch_mask"], dtype=bool)
    idle_rows = np.flatnonzero(dm)
    assert idle_rows.size > 0
    veh_row = int(idle_rows[0])

    req = env.observation_curr["pending_requests"]
    rows = {rid: find_req_row_by_id(req, rid) for rid in accepted_ids}
    assert all(v is not None for v in rows.values())

    dispatch = np.full(env.config.max_pending_requests, env.config.no_action_id, dtype=np.int64)
    dispatch[rows[accepted_ids[0]]] = veh_row
    dispatch[rows[accepted_ids[1]]] = veh_row

    env.step(make_action(env, dispatch=dispatch))

    req2 = env.observation_curr["pending_requests"]
    rows2 = {rid: find_req_row_by_id(req2, rid) for rid in accepted_ids}
    assert all(v is not None for v in rows2.values())

    statuses = [int(req2.iloc[rows2[rid]]["status"]) for rid in accepted_ids]
    n_assigned = sum(s == RequestStatusEnum.ASSIGNED.value for s in statuses)
    assert n_assigned == 1, f"Expected exactly 1 assigned, got {statuses}"

# Run Sim

In [ ]:
def discounted_rewards(rewards: np.ndarray, gamma: float = 0.997) -> np.ndarray:
    """
    Discount rewards using discount factor gamma.
    """
    disc_schedule = exp_decay_half_life(len(rewards), gamma=gamma)
    return rewards * disc_schedule


def get_act_dict(agents: tuple[PricingAgent, DispatchAgent, RepositionAgent], obs: ObservationDict) -> ActionDict:
    price_agent, dispatch_agent, reposition_agent = agents
    prices = price_agent.price(obs)
    dispatch_actions = dispatch_agent.dispatch(obs)
    reposition_actions = reposition_agent.reposition(obs)

    action_agent: ActionDict = {
        "prices": prices,
        "dispatch": dispatch_actions,
        "reposition": reposition_actions,
    }
    return action_agent

In [ ]:
@torch.no_grad()
def run_model_simulation(
    env_curr: RideShareEnv,
    model: RideShareActorCritic,
    num_iter: int = 1,
    gamma: float = GAMMA,
    deterministic: bool = False,
):
    """
    Returns:
      REWARDS: list[episode][t] discounted reward_t
      OBS:     list[episode][t] obs dict (numpy)
      ACT:     list[episode][t] action dict (numpy)
    """
    model.eval()

    REWARDS: list[np.ndarray] = []
    OBS: list[list[dict[str, np.ndarray]]] = []
    ACT: list[list[dict[str, np.ndarray]]] = []
    TERMINATED: list[bool] = []
    TRUNCATED: list[bool] = []

    for i in tqdm(range(num_iter), desc="model rollout"):
        # tqdm.write(f"Starting episode {i+1}/{num_iter}...")
        obs_np, _info = env_curr.reset()
        terminated = False
        truncated = False
        done = False

        rews_raw: list[float] = []
        obs_list: list[dict[str, np.ndarray]] = []
        act_list: list[dict[str, np.ndarray]] = []

        while not done:
            # tqdm.write(f"Step {env_curr.current_step}/{env_curr.config.max_episode_steps}")
            obs_list.append(obs_np)

            obs_t = obs_pd_to_torch(obs_np)

            act_t = model.act(obs_t, deterministic=deterministic)
            act_np = action_torch_to_numpy(act_t)
            env_curr._validate_action(act_np)
            act_list.append(act_np)

            obs_np, reward, terminated, truncated, _info = env_curr.step(act_np)  # type: ignore[arg-type]
            # print(f"Reward: {reward}")
            rews_raw.append(float(reward))
            done = bool(terminated) or bool(truncated)

        rews = discounted_rewards(np.array(rews_raw, dtype=np.float32), gamma=gamma)
        REWARDS.append(rews)
        OBS.append(obs_list)
        ACT.append(act_list)
        TERMINATED.append(terminated)
        TRUNCATED.append(truncated)

    return REWARDS, OBS, ACT, TERMINATED, TRUNCATED

In [ ]:
REWARDS, OBS, ACT, TERMINATED, TRUNCATED = run_model_simulation(env_model, actor_critic_model, num_iter=1)

model rollout:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
TERMINATED, TRUNCATED

([False], [True])

In [ ]:
env_model._rewards

{'penalty_multiple_dispatch_assignment': 0.30000000000000004,
 'penalty_assign_to_unavailable_vehicle': 0.6000000000000001,
 'distance_penalty_xy_normed': 0.0,
 'penalty_rejected': np.float64(-0.0),
 'penalty_expire': np.float64(-0.0),
 'ride_reward_total': np.float64(-18.014681963832118)}

In [ ]:
# import importlib
# import waymo_agent.data_classes.enriched_df_base

# _ = importlib.reload(waymo_agent.data_classes.enriched_df_base)